# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (co2_emissions)
df = pd.read_csv('co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [ ]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


## Task 1 - A line chart showing CO2 per Capita for all countries in the dataset

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [ ]:
# Task 1 — CO₂ per capita for all countries, with one highlighted

# Find the per-capita column without assuming one exact spelling.
per_capita_candidates = [
    "CO2_per_capita", "CO2_per_capita_tonnes", "CO2_per_capita_t",
    "CO2_per_capita (t)", "CO₂_per_capita", "Per_capita"
]
per_capita_col = next(
    (col for col in per_capita_candidates if col in df.columns),
    next((col for col in df.columns if "capita" in col.lower()), None)
)
if per_capita_col is None:
    raise KeyError(
        "Could not identify a CO₂-per-capita column. Available columns: "
        + ", ".join(df.columns)
    )

plot_df = df[["Country", "Year", per_capita_col]].dropna().copy()
plot_df["Year"] = pd.to_numeric(plot_df["Year"], errors="coerce")
plot_df[per_capita_col] = pd.to_numeric(plot_df[per_capita_col], errors="coerce")
plot_df = plot_df.dropna().sort_values("Year")

# Highlight the country with the highest per-capita value in the latest year.
latest_year = plot_df["Year"].max()
latest = plot_df[plot_df["Year"] == latest_year]
highlight_country = latest.loc[latest[per_capita_col].idxmax(), "Country"]
highlight_data = plot_df[plot_df["Country"] == highlight_country]

fig = go.Figure()

# Context lines: all countries in light grey and thin.
for country, group in plot_df.groupby("Country"):
    if country == highlight_country:
        continue
    fig.add_trace(go.Scatter(
        x=group["Year"], y=group[per_capita_col],
        mode="lines", name=str(country),
        line=dict(color="#EEEEEE", width=1),
        hovertemplate=f"{country}<br>Year: %{{x}}<br>CO₂ per capita: %{{y:.2f}}<extra></extra>",
        showlegend=False
    ))

# One highlighted country in colour.
fig.add_trace(go.Scatter(
    x=highlight_data["Year"], y=highlight_data[per_capita_col],
    mode="lines", name=str(highlight_country),
    line=dict(color="#D95F02", width=3),
    hovertemplate=f"{highlight_country}<br>Year: %{{x}}<br>CO₂ per capita: %{{y:.2f}}<extra></extra>",
    showlegend=False
))

last_point = highlight_data.iloc[-1]
fig.add_annotation(
    x=last_point["Year"], y=last_point[per_capita_col],
    text=str(highlight_country), showarrow=False,
    xshift=8, yshift=0, xanchor="left",
    font=dict(size=12, color="#D95F02")
)
fig.update_layout(
    title=dict(
        text=f"{highlight_country} has the highest CO₂ per capita in {int(latest_year)}",
        x=0, xanchor="left"
    ),
    xaxis_title="Year",
    yaxis_title="CO₂ emissions per capita",
    template="plotly_white",
    showlegend=False,
    margin=dict(l=55, r=110, t=85, b=55),
    plot_bgcolor="white",
    paper_bgcolor="white"
)
fig.update_xaxes(showline=False, showgrid=False, zeroline=False)
fig.update_yaxes(showline=False, showgrid=True, gridcolor="#EEEEEE", zeroline=False)
fig.show()

---
## Task 2 — A line chart showing CO2 emissions over time for all Asian countries in the dataset, with one country highlighted.

**same requirements as Task1**

In [ ]:
# Task — Multi-series line with highlight
# Task 2 — CO₂ emissions over time for Asian countries, with one highlighted

# Keep only rows marked as Asia (case-insensitive).
asia_df = df[df["Region"].astype(str).str.strip().str.casefold() == "asia"].copy()
if asia_df.empty:
    raise ValueError(
        "No rows found where Region is 'Asia'. Check the values in df['Region']."
    )

asia_df["Year"] = pd.to_numeric(asia_df["Year"], errors="coerce")
asia_df["CO2_Mt"] = pd.to_numeric(asia_df["CO2_Mt"], errors="coerce")
asia_df = asia_df.dropna(subset=["Country", "Year", "CO2_Mt"]).sort_values("Year")

# Highlight the Asian country with the highest emissions in the latest available year.
latest_year_asia = asia_df["Year"].max()
latest_asia = asia_df[asia_df["Year"] == latest_year_asia]
highlight_country_asia = latest_asia.loc[latest_asia["CO2_Mt"].idxmax(), "Country"]
highlight_asia = asia_df[asia_df["Country"] == highlight_country_asia]

fig_asia = go.Figure()

# All other Asian countries are shown in thin light grey for context.
for country, group in asia_df.groupby("Country"):
    if country == highlight_country_asia:
        continue
    fig_asia.add_trace(go.Scatter(
        x=group["Year"], y=group["CO2_Mt"],
        mode="lines", name=str(country),
        line=dict(color="#EEEEEE", width=1),
        hovertemplate=f"{country}<br>Year: %{{x}}<br>CO₂ emissions: %{{y:.1f}} Mt<extra></extra>",
        showlegend=False
    ))

# Highlight one country and label it directly at the end of the line.
fig_asia.add_trace(go.Scatter(
    x=highlight_asia["Year"], y=highlight_asia["CO2_Mt"],
    mode="lines", name=str(highlight_country_asia),
    line=dict(color="#0072B2", width=3),
    hovertemplate=f"{highlight_country_asia}<br>Year: %{{x}}<br>CO₂ emissions: %{{y:.1f}} Mt<extra></extra>",
    showlegend=False
))

last_asia_point = highlight_asia.iloc[-1]
fig_asia.add_annotation(
    x=last_asia_point["Year"], y=last_asia_point["CO2_Mt"],
    text=str(highlight_country_asia), showarrow=False,
    xshift=8, xanchor="left",
    font=dict(size=12, color="#0072B2")
)
fig_asia.update_layout(
    title=dict(
        text=f"{highlight_country_asia} has the highest CO₂ emissions in Asia in {int(latest_year_asia)}",
        x=0, xanchor="left"
    ),
    xaxis_title="Year",
    yaxis_title="CO₂ emissions (Mt)",
    template="plotly_white",
    showlegend=False,
    margin=dict(l=55, r=110, t=85, b=55),
    plot_bgcolor="white",
    paper_bgcolor="white"
)
fig_asia.update_xaxes(showline=False, showgrid=False, zeroline=False)
fig_asia.update_yaxes(showline=False, showgrid=True, gridcolor="#EEEEEE", zeroline=False)
fig_asia.show()
